# CNN Emotion Classifier

This notebook contains the full CNN workflow in one place.

In [ ]:
from pathlib import Path
import csv
import json
import os
import random

import librosa
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_DIR = Path(".")
CACHE_DIR = Path("cnn_cache")
RESULTS_DIR = Path("cnn_results")

SAMPLE_RATE = 22050
DURATION = 3.0
OFFSET = 0.0
TRIM_SILENCE = True
TRIM_TOP_DB = 30

N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256
FMIN = 50
FMAX = 8000
TOP_DB = 80

TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
PATIENCE = 10

# Keep as None for the full dataset. Use a small number like 8 for a quick test.
MAX_FILES_PER_CLASS = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Find WAV files and labels

In [ ]:
RAVDESS_EMOTIONS = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

IGNORE_DIRS = {
    ".venv",
    ".ipynb_checkpoints",
    "__pycache__",
    "cnn_cache",
    "cnn_results",
}


def emotion_from_name(path):
    parts = path.stem.split("-")
    if len(parts) >= 3:
        return RAVDESS_EMOTIONS.get(parts[2], "unknown")
    return "unknown"


def actor_from_name(path):
    parts = path.stem.split("-")
    if len(parts) >= 7:
        return parts[6]
    return "unknown"


def find_wav_files(folder):
    wav_files = []
    for root, dirs, files in os.walk(folder):
        dirs[:] = [d for d in dirs if d.lower() not in IGNORE_DIRS]
        for file_name in files:
            path = Path(root) / file_name
            if path.suffix.lower() == ".wav":
                wav_files.append(path)
    return sorted(wav_files)


records = []
for path in find_wav_files(INPUT_DIR):
    emotion = emotion_from_name(path)
    if emotion != "unknown":
        records.append({"path": path, "emotion": emotion, "actor": actor_from_name(path)})

if MAX_FILES_PER_CLASS is not None:
    limited = []
    for emotion in sorted({r["emotion"] for r in records}):
        group = [r for r in records if r["emotion"] == emotion]
        random.shuffle(group)
        limited.extend(group[:MAX_FILES_PER_CLASS])
    records = sorted(limited, key=lambda r: str(r["path"]))

print("Total labelled WAV files:", len(records))
counts = {emotion: sum(r["emotion"] == emotion for r in records) for emotion in sorted({r["emotion"] for r in records})}
print(counts)
records[:3]


## Create log-mel features

In [ ]:
def fix_length(y, sr, duration):
    target_len = int(sr * duration)
    if len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)), mode="constant")
    return y[:target_len]


def make_log_mel(path):
    y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True, offset=OFFSET)

    if TRIM_SILENCE:
        y, _ = librosa.effects.trim(y, top_db=TRIM_TOP_DB)

    y = fix_length(y, sr, DURATION)
    y = librosa.effects.preemphasis(y)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    return mel_db.astype(np.float32)


def cache_path(record):
    return CACHE_DIR / record["emotion"] / f"{record['path'].stem}.npy"


for i, record in enumerate(records, start=1):
    out_path = cache_path(record)
    if not out_path.exists():
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, make_log_mel(record["path"]))
    record["feature_path"] = out_path

    if i == 1 or i == len(records) or i % 100 == 0:
        print(f"Cached/checked {i}/{len(records)}")

sample = np.load(records[0]["feature_path"])
print("Feature shape:", sample.shape)
print("Feature range:", float(sample.min()), "to", float(sample.max()))


## Split the dataset

In [ ]:
labels = [r["emotion"] for r in records]
train_val, test_records = train_test_split(
    records,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
    stratify=labels,
)

train_val_labels = [r["emotion"] for r in train_val]
relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
train_records, val_records = train_test_split(
    train_val,
    test_size=relative_val_size,
    random_state=SEED,
    shuffle=True,
    stratify=train_val_labels,
)

classes = sorted({r["emotion"] for r in records})
class_to_idx = {name: i for i, name in enumerate(classes)}
idx_to_class = {i: name for name, i in class_to_idx.items()}


def count_split(split):
    return {emotion: sum(r["emotion"] == emotion for r in split) for emotion in classes}

print("Classes:", classes)
print("Train/val/test:", len(train_records), len(val_records), len(test_records))
print("Train:", count_split(train_records))
print("Val:", count_split(val_records))
print("Test:", count_split(test_records))


## Dataset and dataloaders

In [ ]:
def get_mean_std(split):
    total = 0.0
    total_sq = 0.0
    count = 0
    for record in split:
        x = np.load(record["feature_path"]).astype(np.float64)
        total += x.sum()
        total_sq += np.square(x).sum()
        count += x.size
    mean = total / count
    std = np.sqrt(max(total_sq / count - mean**2, 1e-8))
    return float(mean), float(std)


MEAN, STD = get_mean_std(train_records)
print("Mean/std:", MEAN, STD)


class MelDataset(Dataset):
    def __init__(self, split, augment=False):
        self.split = split
        self.augment = augment

    def __len__(self):
        return len(self.split)

    def __getitem__(self, index):
        record = self.split[index]
        x = np.load(record["feature_path"]).astype(np.float32)
        x = (x - MEAN) / STD
        x = torch.from_numpy(x).unsqueeze(0)

        if self.augment:
            x = self.mask_random_parts(x)

        y = torch.tensor(class_to_idx[record["emotion"]], dtype=torch.long)
        return x, y

    def mask_random_parts(self, x):
        x = x.clone()
        _, freq, time = x.shape

        if random.random() < 0.5:
            width = random.randint(0, min(12, freq))
            start = random.randint(0, freq - width) if width > 0 else 0
            x[:, start:start + width, :] = 0

        if random.random() < 0.5:
            width = random.randint(0, min(24, time))
            start = random.randint(0, time - width) if width > 0 else 0
            x[:, :, start:start + width] = 0

        return x


train_loader = DataLoader(MelDataset(train_records, augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MelDataset(val_records), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MelDataset(test_records), batch_size=BATCH_SIZE, shuffle=False)

batch_x, batch_y = next(iter(train_loader))
print("Batch X:", batch_x.shape)
print("Batch y:", batch_y.shape)


## CNN model

In [ ]:
class SimpleEmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            self.block(1, 32, 0.05),
            self.block(32, 64, 0.10),
            self.block(64, 128, 0.15),
            self.block(128, 256, 0.20),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, num_classes),
        )

    def block(self, in_channels, out_channels, dropout):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SimpleEmotionCNN(num_classes=len(classes)).to(DEVICE)
print(model)


## Train

In [ ]:
def class_weights(split):
    counts = count_split(split)
    total = sum(counts.values())
    weights = []
    for emotion in classes:
        weights.append(total / (len(classes) * counts[emotion]))
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


criterion = nn.CrossEntropyLoss(weight=class_weights(train_records))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)


def run_epoch(loader, training):
    model.train(training)
    losses = []
    preds = []
    actuals = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(x)
            loss = criterion(logits, y)

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 3.0)
                optimizer.step()

        losses.append(loss.item())
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        actuals.extend(y.detach().cpu().tolist())

    return float(np.mean(losses)), float(accuracy_score(actuals, preds)), actuals, preds


RESULTS_DIR.mkdir(exist_ok=True)
best_val_acc = 0.0
best_epoch = 0
no_improve = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, training=False)
    scheduler.step(val_acc)

    lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": epoch,
        "learning_rate": lr,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"lr={lr:.1e} "
        f"train_acc={train_acc:.3f} val_acc={val_acc:.3f} "
        f"train_loss={train_loss:.3f} val_loss={val_loss:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "class_to_idx": class_to_idx,
            "mean": MEAN,
            "std": STD,
            "settings": {
                "sample_rate": SAMPLE_RATE,
                "duration": DURATION,
                "n_mels": N_MELS,
                "n_fft": N_FFT,
                "hop_length": HOP_LENGTH,
                "fmin": FMIN,
                "fmax": FMAX,
            },
        }, RESULTS_DIR / "best_model.pt")
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print("Early stopping")
        break

print("Best validation accuracy:", best_val_acc, "at epoch", best_epoch)


## Test and save results

In [ ]:
checkpoint = torch.load(RESULTS_DIR / "best_model.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc, y_true, y_pred = run_epoch(test_loader, training=False)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

report = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
matrix = confusion_matrix(y_true, y_pred)

with (RESULTS_DIR / "training_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)

with (RESULTS_DIR / "classification_report.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

with (RESULTS_DIR / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["actual/predicted", *classes])
    for label, row in zip(classes, matrix):
        writer.writerow([label, *row.tolist()])

summary = {
    "classes": classes,
    "train_count": len(train_records),
    "val_count": len(val_records),
    "test_count": len(test_records),
    "best_val_accuracy": best_val_acc,
    "best_epoch": best_epoch,
    "test_accuracy": test_acc,
    "test_loss": test_loss,
    "model_path": str(RESULTS_DIR / "best_model.pt"),
}

with (RESULTS_DIR / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

summary


## Predict one file

In [ ]:
def predict_wav(path):
    checkpoint = torch.load(RESULTS_DIR / "best_model.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    x = make_log_mel(Path(path))
    x = (x - checkpoint["mean"]) / checkpoint["std"]
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1).squeeze(0).cpu().numpy()

    return sorted(zip(checkpoint["classes"], probs), key=lambda item: item[1], reverse=True)


sample_path = records[0]["path"]
print("File:", sample_path)
print("Actual:", emotion_from_name(sample_path))
for emotion, probability in predict_wav(sample_path):
    print(f"{emotion:10s} {probability:.3f}")
summary
